# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [12]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")
df.head()

,STATUS,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS,hot_F,hot_M,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,0,1,1,1,0,0,2.475055,0.940015,-1.073197,-0.553082,-0.196176,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,0,1,1,0,0,0,-0.692823,-1.291452,0.400897,-0.553082,-0.196176,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,0,0,1,0,1,1,0.891116,-0.734351,-0.428281,-0.553082,-1.267090,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,0,0,1,0,0,0,1.026882,-1.524756,0.891830,-0.553082,-1.267090,True,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
4,0,1,1,1,1,1,0.891116,-0.206943,0.558774,-0.553082,-0.196176,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [13]:
# Variáveis independentes
X = df.drop(columns=[TARGET])

# Variável Alvo
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

*Modelo KNN*

In [18]:
from sklearn.metrics import mean_absolute_error, r2_score


knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

accuracy = knn.score(X_test, y_test)

# Erro médio absoluto
MAE = mean_absolute_error(y_test, y_pred_knn)

r2 = r2_score(y_test, y_pred_knn) # qto mais próximo de 1, melhor

print('MAE',MAE)
print('r²',r2)

print("Acurácia: {:.2f}".format(round(accuracy,4)))

MAE 0.04523107177974434
r² -0.09273100999719674
Acurácia: 0.95


In [35]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


modelos_desbalanceados = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=3)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
}


modelos_balanceados = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=3)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
}

## 3. Validação cruzada

In [42]:
from sklearn.model_selection import cross_val_score

print("Modelos desbalanceados")
for nome, modelo in modelos_desbalanceados.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')

print("\nModelos balanceados")
for nome, modelo in modelos_balanceados.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')
    




Modelos desbalanceados
F1: 0.0194	 Accuracy: 0.9510	KNN
F1: 0.0000	 Accuracy: 0.9564	Regressão Logística
F1: 0.0000	 Accuracy: 0.9281	Floresta Aleatória
F1: 0.0241	 Accuracy: 0.9109	Árvore de Decisão

Modelos balanceados
F1: 0.0194	 Accuracy: 0.9510	KNN
F1: 0.0848	 Accuracy: 0.6045	Regressão Logística
F1: 0.0282	 Accuracy: 0.9244	Floresta Aleatória
F1: 0.0277	 Accuracy: 0.8800	Árvore de Decisão


## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_